## Remote ID Sppoofing Attack

In [ ]:
import pickle
from typing import Literal

from simulator import Oracle, Simulator
from simulator.config import DATA_PATH, Color, Model
from simulator.entities import SimGCS, SimVehicle
from simulator.helpers import SimProcess, clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.planner import AutoPlan
from simulator.visualizer import Gazebo, GazMarker

clean()


## Simulation Positions

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

base_homes = ENUPose.list([(-10, 0, 0, 0), (0, -10, 0, 0)])
base_paths = [
    ENU.list([(0, 0, 0), (0, 0, 5), (20, 0, 5)]),
    ENU.list([(0, 0, 0), (0, 0, 5)]),
]


## Oracle

In [ ]:
orac = Oracle()

## Create Vehicles

In [ ]:
sysids = [1, 255]  # attacker sysid is (temporally) being hardcoded to 255
colors = [Color.GREEN, Color.RED]
lands = [True, False]
model = Model.IRIS

mission_folder = DATA_PATH / "missions"
mission_folder.mkdir(parents=True, exist_ok=True)

for sysid, base_home, base_path, color, land in zip(
    sysids, base_homes, base_paths, colors, lands, strict=True
):
    mission_path = str(mission_folder / f"mission_{sysid}.waypoints")
    auto_plan = AutoPlan.from_relative_path(
        name="simple_auto_plan",
        sysid=sysid,
        gra_origin=gra_origin,
        relative_home=base_home,
        relative_path=base_path,
        land=land,
        navigation_speed=1.0,
        mission_path=mission_path,
        firmware=model.firmware,
    )

    veh = SimVehicle.from_relative(
        sysid=sysid,
        plan=auto_plan,
        color=color,
        enu_origin=enu_origin,
        relative_home=base_home,
        relative_path=base_path,
        model=model,
    )
    orac.add_vehicle(veh)


## Scenario configurarion

In [ ]:
fake_pos = ENU(x=0, y=0, z=5)
true_pos = enu_origin.to_abs(base_homes[1]).to_abs(base_paths[1][0]).unpose()
avoidance_method: Literal["stop", "naive"] = "naive"

safety_radius = 5.0  # meters
radar_radius = 10.0  # meters

# This is temporary to visualize the scenario
with open(DATA_PATH / "fake_position.pkl", "wb") as f:
    pickle.dump(fake_pos, f)


## Gazebo

In [ ]:
gaz = Gazebo(gra_origin, world_path="simulator/visualizer/gazebo/worlds/runway.world")

origin_gaz = GazMarker(
    name="origin", group="origin", pos=enu_origin.unpose(), color=Color.WHITE
)

true_marker = GazMarker(
    name="true_pos", group="true_pos", pos=true_pos, color=Color.RED
)

fake_marker = GazMarker(
    name="fake_pos", group="fake_pos", pos=fake_pos, color=Color.ORANGE
)

avoid_zone = GazMarker(
    name="avoid_zone",
    group="avoid_zone",
    pos=fake_pos,
    color=Color.RED,
    radius=safety_radius,
    alpha=0.85,
)

radar_zone = GazMarker(
    name="radar_zone",
    group="radar_zone",
    pos=true_pos,
    color=Color.ORANGE,
    radius=radar_radius,
    alpha=0.9,
)

for marker in [origin_gaz, true_marker, fake_marker, avoid_zone, radar_zone]:
    gaz.markers.append(marker)


## Simulator

In [ ]:
simulator = Simulator(
    oracle=orac,
    visualizer=gaz,
    terminals=[SimProcess.LOGIC],
    verbose=1,
)

simulator.preview()


## Run

In [ ]:
simulator.run(timeout=180)


## Ground truth: real vs transmitted

The Oracle only *receives* Remote ID, which the attacker (sysid 255) spoofs, so
its live track is the **transmitted** position (dots). `truth=` overlays the
**real** trajectory (line), reconstructed by the Oracle from the ground-truth
logs — the spoof is the gap between the two.

In [ ]:
# Transmitted only (spoofed) — the Oracle's raw Remote ID view
orac.plot_trajectories()

# Real path (line) overlaid on the transmitted RID (dots)
orac.plot_trajectories(truth=True)

# Just the attacker's real vs spoofed position
orac.plot_trajectories(truth=[255]);